# Atomic structures: build, select, edit and validate

Worked examples of the `discuspdf.Structure` API: construction, validation,
select/edit, distances, and wrapping.

In [1]:
import numpy as np
from discuspdf import Lattice, Structure

## GaAs conventional cell

Zinc blende, a = 4.53 Å. Ga at (0, 0, 0), As at (0.25, 0.25, 0.25).

In [2]:
a = 4.53
lat = Lattice.from_parameters(a, a, a, 90.0, 90.0, 90.0)
gaas = Structure(
    lat,
    np.array(["Ga", "As"]),
    np.array([[0.0, 0.0, 0.0], [0.25, 0.25, 0.25]]),
    model="configuration",
)
print(gaas)
print(f"counts: {gaas.counts()}")
print(f"composition: {gaas.composition()}")
print(f"formula: {gaas.formula()}")
print(f"atomic numbers: {gaas.atomic_numbers()}")

Structure(n=2, formula=GaAs, model='configuration', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=4.53 Å, b=4.53 Å, c=4.53 Å, alpha=90°, beta=90°, gamma=90°)
counts: {'Ga': 1, 'As': 1}
composition: {'Ga': 1.0, 'As': 1.0}
formula: GaAs
atomic numbers: [31 33]


In [3]:
# Ga-As distance
d = gaas.distance(0, 1)
print(f"Ga-As distance: {d:.6f} Å  (expected {a * np.sqrt(3) / 4:.6f})")

Ga-As distance: 1.961548 Å  (expected 1.961548)


## Cu/Au average model

FCC, a = 4.0 Å. Mixed site: Cu 0.75, Au 0.25.

In [4]:
a_cu = 4.0
lat_cu = Lattice.from_parameters(a_cu, a_cu, a_cu, 90.0, 90.0, 90.0)
cupau = Structure(
    lat_cu,
    np.array(["Cu", "Au"]),
    np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]),
    model="average",
    occupancy=np.array([0.75, 0.25]),
    site_label=np.array(["S1", "S1"]),
)
print(cupau)
print(f"composition: {cupau.composition()}")
print(f"formula: {cupau.formula()}")

Structure(n=2, formula=Cu0.75Au0.25, model='average', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=4 Å, b=4 Å, c=4 Å, alpha=90°, beta=90°, gamma=90°)
composition: {'Cu': 0.75, 'Au': 0.25}
formula: Cu0.75Au0.25


## Select and edit

Functional edits return new `Structure` instances; the original is untouched.

In [5]:
# Select Ga only
ga_only = gaas.select(species="Ga")
print(f"Select Ga only: {ga_only}")

# Edit: move As
gaas2 = gaas.with_frac(np.array([[0.0, 0.0, 0.0], [0.3, 0.25, 0.25]]))
d2 = gaas2.distance(0, 1)
print(f"After moving As: Ga-As distance = {d2:.6f} Å")

# Original unchanged
d_orig = gaas.distance(0, 1)
print(f"Original Ga-As distance: {d_orig:.6f} Å (unchanged)")

Select Ga only: Structure(n=1, formula=Ga, model='configuration', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=4.53 Å, b=4.53 Å, c=4.53 Å, alpha=90°, beta=90°, gamma=90°)
After moving As: Ga-As distance = 2.100475 Å
Original Ga-As distance: 1.961548 Å (unchanged)


## Wrap

`wrapped()` returns positions in [0, 1) for fully periodic structures.

In [6]:
gaas_wrapped = gaas.wrapped()
print(f"Wrapped: {gaas_wrapped.frac}")

Wrapped: [[0.   0.   0.  ]
 [0.25 0.25 0.25]]


## Validation

Common validation errors:

In [7]:
# Configuration with occupancy < 1 raises
try:
    Structure(lat, np.array(["Ga"]), np.array([[0.25, 0.25, 0.25]]),
              model="configuration", occupancy=np.array([0.5]))
except ValueError as e:
    print(f"Configuration occ < 1: {e}")

# Site occupancy sum > 1 raises
try:
    Structure(lat, np.array(["Cu", "Au"]),
              np.array([[0.25, 0.25, 0.25], [0.25, 0.25, 0.25]]),
              model="average", occupancy=np.array([0.7, 0.5]),
              site_label=np.array(["S1", "S1"]))
except ValueError as e:
    print(f"Site occ sum > 1: {e}")

Configuration occ < 1: configuration model: all occupancies must be exactly 1.0; for partial occupancies use model='average'
Site occ sum > 1: average model: site 'S1' in cell (0, 0, 0) has total occupancy 1.200000 > 1 + 1e-6
